# Análise de Replay de Mercado Estático

Este notebook gera um gráfico de candles para um dia específico e, em seguida, analisa cada candle para gerar um relatório de todas as sugestões de operação que o modelo de IA teria fornecido.

**Funcionalidades:**
- Gera um gráfico completo para um dia ou para um período parcial do dia.
- Exibe o gráfico com candles de M5 e as médias móveis (SMA9, EMA21, EMA50, EMA200).
- Analisa cada candle do período para gerar sinais de Compra/Venda com base no modelo treinado.
- Apresenta um relatório final com todas as operações sugeridas, incluindo candle, tipo, preço de entrada e preço de stop.

### Passo 1: Configuração da Análise

**Ação:** Defina as variáveis `TICKER` e `REPLAY_DATETIME_STR`.
- `REPLAY_DATETIME_STR`: Pode ser apenas a data (`'YYYY-MM-DD'`) para analisar o dia todo, ou a data e hora (`'YYYY-MM-DD HH:MM'`) para analisar o dia até aquele momento.

In [ ]:
# --- PARÂMETROS DE ENTRADA ---
TICKER = "WDO$"  # Ativo para o replay (deve estar no main.yaml)
REPLAY_DATETIME_STR = "2025-10-10" # Data ou Data e Hora
# ---------------------------

print(f"Configurado para análise do ativo '{TICKER}' em '{REPLAY_DATETIME_STR}'.")

### Passo 2: Importações e Preparação do Ambiente

In [ ]:
import yaml
import logging
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
import pytz
import MetaTrader5 as mt5
import mplfinance as mpf
import sys

# Adiciona a pasta 'src' ao path
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.strategies.lstm import KerasLSTMWrapper

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Carrega as configurações
with open(project_root / "configs/main.yaml", "r") as file:
    config = yaml.safe_load(file)

# Encontra a configuração específica do ativo
asset_config = next((asset for asset in config['assets'] if asset['ticker'] == TICKER), None)
if not asset_config:
    raise ValueError(f"Ticker '{TICKER}' não encontrado em configs/main.yaml.")

### Passo 3: Geração do Gráfico e Relatório de Sinais

In [ ]:
def generate_static_replay_and_report(ticker, datetime_str, asset_config):
    """Busca dados, gera o gráfico completo e o relatório de sinais."""
    # 1. Processar data e hora
    try:
        end_dt_obj = datetime.strptime(datetime_str, '%Y-%m-%d %H:%M')
        start_dt_obj = end_dt_obj.replace(hour=0, minute=0, second=0)
    except ValueError:
        start_dt_obj = datetime.strptime(datetime_str, '%Y-%m-%d')
        end_dt_obj = start_dt_obj.replace(hour=23, minute=59, second=59)

    timezone = pytz.timezone("Etc/UTC")
    start_time_utc = timezone.localize(start_dt_obj)
    end_time_utc = timezone.localize(end_dt_obj)

    # 2. Buscar dados de candles M5
    logging.info(f"Conectando ao MT5 para buscar candles de M5 para {ticker}...")
    if not mt5.initialize():
        logging.error(f"Falha na inicialização do MT5: {mt5.last_error()}")
        return
    
    rates = mt5.copy_rates_range(ticker, mt5.TIMEFRAME_M5, start_time_utc, end_time_utc)
    mt5.shutdown()

    if rates is None or len(rates) == 0:
        logging.error("Nenhum dado de candle encontrado para o período.")
        return
    
    candles_df = pd.DataFrame(rates)
    candles_df['time'] = pd.to_datetime(candles_df['time'], unit='s')
    candles_df.set_index('time', inplace=True)
    candles_df.rename(columns={'tick_volume': 'volume'}, inplace=True)
    logging.info(f"{len(candles_df)} candles de M5 carregados.")

    # 3. Calcular indicadores
    candles_df['sma9'] = candles_df['close'].rolling(window=9).mean()
    candles_df['ema21'] = candles_df['close'].ewm(span=21, adjust=False).mean()
    candles_df['ema50'] = candles_df['close'].ewm(span=50, adjust=False).mean()
    candles_df['ema200'] = candles_df['close'].ewm(span=200, adjust=False).mean()

    # 4. Gerar Gráfico
    chart_style = mpf.make_mpf_style(base_mpf_style='default', marketcolors=mpf.make_marketcolors(up='g', down='r'), facecolor='white', gridstyle='-')
    addplots = [
        mpf.make_addplot(candles_df['sma9'], color='red'),
        mpf.make_addplot(candles_df['ema21'], color='blue'),
        mpf.make_addplot(candles_df['ema50'], color='orange'),
        mpf.make_addplot(candles_df['ema200'], color='black'),
    ]
    
    logging.info("Gerando gráfico estático...")
    mpf.plot(candles_df, type='candle', style=chart_style, addplot=addplots, figsize=(20, 8), title=f"Replay Estático - {ticker} - {start_dt_obj.date()}")
    
    # 5. Gerar Relatório de Sinais
    logging.info("Gerando relatório de sinais de operação...")
    models_dir = project_root / config["global_settings"]["model_directory"]
    model_path = models_dir / f"{ticker}_prod_model.keras"
    scaler_path = models_dir / f"{ticker}_prod_scaler.joblib"

    if not model_path.exists():
        logging.error(f"Modelo para {ticker} não encontrado. Execute o train_model.py primeiro.")
        return
        
    model = KerasLSTMWrapper.load_model(str(model_path), str(scaler_path))
    strategy_module = importlib.import_module(f"src.strategies.{asset_config['strategy_module']}")
    StrategyClass = getattr(strategy_module, asset_config['strategy_name'])
    strategy = StrategyClass()

    suggestions = []
    # Adiciona dados históricos suficientes para o lookback
    hist_start = start_time_utc - timedelta(days=5) # Busca 5 dias de histórico para garantir lookback
    mt5.initialize()
    hist_rates = mt5.copy_rates_range(ticker, mt5.TIMEFRAME_M5, hist_start, end_time_utc)
    mt5.shutdown()
    hist_df = pd.DataFrame(hist_rates)
    hist_df['time'] = pd.to_datetime(hist_df['time'], unit='s')
    hist_df.set_index('time', inplace=True)
    hist_df.rename(columns={'tick_volume': 'volume'}, inplace=True)

    # Itera sobre cada candle do dia de replay
    for i in range(len(candles_df)):
        current_candle_time = candles_df.index[i]
        
        # Pega todos os dados até o candle atual para a predição
        data_for_prediction = hist_df[hist_df.index <= current_candle_time]
        
        featured_data = strategy.define_features(data_for_prediction)
        X_live = featured_data[strategy.get_feature_names()].dropna()

        if len(X_live) > model.lookback:
            signal = model.predict(X_live)[-1]
            
            # Calcula preço de entrada e stop
            entry_price = candles_df['close'].iloc[i]
            stop_loss_pct = asset_config['trading_rules']['stop_loss_pct']
            stop_price = entry_price * (1 - stop_loss_pct) if signal == 1 else entry_price * (1 + stop_loss_pct)
            
            suggestions.append({
                "Candle de Entrada": current_candle_time.strftime('%Y-%m-%d %H:%M'),
                "Tipo de Operação": "Compra" if signal == 1 else "Venda",
                "Preço Sugerido": f"{entry_price:.2f}",
                "Preço de Stop": f"{stop_price:.2f}"
            })
    
    if suggestions:
        report_df = pd.DataFrame(suggestions)
        display(report_df)
    else:
        print("Nenhuma sugestão de operação foi gerada para este período.")

# Executa a função principal
generate_static_replay_and_report(TICKER, REPLAY_DATETIME_STR, asset_config)
